# Notebook 04 TA Answer Key: Supervised Learning and Assignment 3 Bridge

This TA-only notebook matches the student Session 4 notebook. It includes complete code, expected checkpoints, teaching notes, and suggested hints. Use hints first; reveal full code only after students have tried a small example or explained their plan.


## Learning Goals

- Explain supervised learning in plain language.
- Split data into training and test sets.
- Train and evaluate simple classification models.
- Recognize common model-evaluation mistakes.
- Connect model practice to Assignment 3's dataset workflow.
- Load, clean, summarize, visualize, and cluster a tabular dataset.


## Warm-up Answer Notes

1. A model might predict a category such as flower species, a numeric value such as price, or a binary outcome such as yes/no.
2. Input features are the information the model receives; the target is the answer it learns to predict.
3. Testing on unseen examples checks whether the model learned a pattern instead of memorizing training rows.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.cluster import KMeans


# Part 1: Supervised Learning

Supervised learning uses examples with known answers.

- `X`: input features.
- `y`: target or answer.
- Classification predicts categories.
- Regression predicts numbers.

Today we focus on classification.


## Section 1: Load a Classification Dataset


In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")

iris_df = X.copy()
iris_df["species_name"] = y.map(dict(enumerate(iris.target_names)))
iris_df.head()


### Exercise 1.1: Name `X` and `y`

**Answer notes:**

- `X` contains the four measurement columns: sepal length, sepal width, petal length, and petal width.
- `y` stores the species label as numbers: `0`, `1`, or `2`.
- One row of `X` is one flower's measurements; the matching value of `y` is that flower's species.

**Hint to give first:** Ask students, "Which columns would be available before we know the answer? Which column is the answer?"


## Section 2: Train/Test Split

Train on one part of the data, then test on examples the model did not see during training.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=1,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))


**What does `stratify=y` mean?**

For a classification problem, `y` contains the class labels. `stratify=y` tells `train_test_split` to keep about the same class proportions in the training set and the test set.

Example: the iris dataset has three species. If each species is about one-third of the full dataset, stratifying helps the test set stay close to one-third of each species too. This makes the test set a fairer check of the model.


### Exercise 2.1: Explain the Train/Test Split Inputs

**Answer notes:**

- `X`: the feature table the model uses as input.
- `y`: the target labels the model learns to predict.
- `test_size=0.25`: hold out 25% of rows for testing.
- `random_state=1`: make the random split repeatable.
- `stratify=y`: keep the species/class proportions similar in train and test sets.

**Common mistake:** Students often say `random_state` improves accuracy. It does not; it only makes the same split happen again.


## Section 3: Model 1 - K-Nearest Neighbors

KNN predicts a class by looking at nearby training examples.


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

knn_predictions = knn.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_predictions)

print("KNN accuracy:", knn_accuracy)


### Exercise 3.1: Try Different Values of `k`

Train KNN models with `k = 1`, `3`, `5`, and `9`. Store each test accuracy in a small results table.


In [ ]:
knn_results = []

for k in [1, 3, 5, 9]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    knn_results.append({"k": k, "accuracy": accuracy})

pd.DataFrame(knn_results)


**TA checkpoint:** With this split, `k = 1`, `3`, and `9` score about `0.974`; `k = 5` scores about `0.947`. Bigger `k` does not automatically win.


## Section 4: Model 2 - Decision Tree

Decision trees split the data using feature-based questions.


In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=1)
tree.fit(X_train, y_train)

tree_predictions = tree.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print("Decision tree accuracy:", tree_accuracy)


### Exercise 4.1: Compare the Two Models

Create a DataFrame with one row for KNN and one row for the decision tree. Include model name and test accuracy.


In [ ]:
model_results = [
    {"model": "KNN (k=5)", "test_accuracy": knn_accuracy},
    {"model": "Decision tree (max_depth=3)", "test_accuracy": tree_accuracy},
]

pd.DataFrame(model_results)


**TA checkpoint:** On this split, the decision tree scores about `0.974`, while the default KNN example scores about `0.947`. Emphasize "on this split," not "always better."


## Section 5: Evaluation

Accuracy is useful, but it is not the whole story. A confusion matrix shows which classes are getting mixed up.


### Common Evaluation Metrics

| Scenario | Useful metrics | What the metric helps answer |
| --- | --- | --- |
| Balanced classification | accuracy | What fraction of predictions were correct? |
| Imbalanced classification | precision, recall, F1 | Is the model missing rare cases or creating too many false alarms? |
| False positives are costly | precision | When the model predicts the positive class, how often is it right? |
| False negatives are costly | recall | Of all real positive cases, how many did the model find? |
| Probability or ranking task | ROC-AUC | Does the model rank positive examples above negative examples? |
| Numeric prediction | MAE, MSE, RMSE, R² | How far are predictions from the real numbers? |
| Clustering with no labels | inertia, silhouette score, plots | Are the groups compact, separated, and interpretable? |


### Confusion Matrix Guide

A confusion matrix compares true labels with predicted labels.

- Rows are the true labels.
- Columns are the predicted labels.
- One cell reads as: true row class predicted as column class.
- The diagonal cells are correct predictions.
- Off-diagonal cells are mistakes.

For a binary problem, precision focuses on false positives and recall focuses on false negatives.


In [ ]:
confusion = confusion_matrix(y_test, tree_predictions)
confusion_df = pd.DataFrame(
    confusion,
    index=iris.target_names,
    columns=iris.target_names,
)

confusion_df


In [ ]:
print(classification_report(
    y_test,
    tree_predictions,
    target_names=iris.target_names,
    zero_division=0,
))


### Exercise 5.1: Read Matrix and Choose a Metric

**Answer notes:**

1. With the current split, the tree has one mistake: one true `virginica` is predicted as `versicolor`.
2. If false positives are costly, watch precision most closely.
3. If false negatives are costly, watch recall most closely.
4. Careful sentence: "On this test split, the decision tree correctly classified 37 of 38 flowers, but it confused one virginica flower for versicolor."

**Hint to give first:** Ask students to pick one off-diagonal cell and read it as "true row class predicted as column class."


# Part 2: Assignment 3 Bridge

Assignment 3 is not mainly about supervised learning. It asks students to work with a real tabular dataset: load it, clean it, summarize it, visualize it, and apply clustering.

This bridge section uses the same Kaggle motorcycle dataset students need for Assignment 3: `data/all_bikez_curated.csv`.

**TA note:** The student notebook intentionally scaffolds this section. Use the complete code below as an answer key, but guide students with inspection/decision questions first.


## Section 6: Assignment 3 Workflow Map

Assignment 3 asks for this kind of workflow:

1. Load a CSV with pandas.
2. Inspect rows, columns, and data types.
3. Clean missing values or text-like numeric columns.
4. Select numeric columns.
5. Use `describe()` and `corr()`.
6. Filter rows with multiple conditions.
7. Make clear visualizations.
8. Apply K-means clustering to numeric data.
9. Interpret results cautiously.


## Section 7: Load the Assignment 3 Dataset

The real Assignment 3 dataset is already in the course `data/` folder.

Use `low_memory=False` so pandas reads mixed columns more consistently. Loading the file is setup; the analysis decisions still come from the student.


In [ ]:
DATA_PATH = "data/all_bikez_curated.csv"

motorcycle_info_raw = pd.read_csv(DATA_PATH, low_memory=False)

print("Rows and columns:", motorcycle_info_raw.shape)
motorcycle_info_raw.head()


### Exercise 7.1: Inspect Before You Decide

Student prompt:

1. How many rows and columns are in the raw dataset?
2. Which columns look numeric? Which columns are text/categorical?
3. Which columns have many missing values?
4. Which numeric-looking columns might need cleaning before analysis?

Useful tools: `head()`, `shape`, `columns`, `info()`, `isna().sum()`, `describe()`, and `corr()`.

**TA hint:** Ask students to name what they learned from each inspection command before they clean anything.


In [ ]:
first_rows = motorcycle_info_raw.head()
print("Shape:", motorcycle_info_raw.shape)
print("Columns:", list(motorcycle_info_raw.columns))
motorcycle_info_raw.info()

first_rows


**TA checkpoint:** The raw dataset has `38,472` rows and `28` columns. If students cannot load it, check that they are running from the repository root or that the `data/` folder exists.


## Section 8: Clean and Select Numeric Columns

Student decision questions:

1. Which column needs conversion?
2. What values should become missing after conversion?
3. How much data would be removed if missing values are dropped?
4. Which columns should be used for numeric summaries or clustering?

**TA solution note:** The worked solution below cleans `Stroke (mm)`, drops missing rows for this practice flow, and then selects numeric columns. Accept other defensible cleaning decisions if students explain the tradeoff.


In [ ]:
motorcycle_info = motorcycle_info_raw.copy()

# Clean Stroke (mm) if it exists and contains commas or text-like values.
if "Stroke (mm)" in motorcycle_info.columns:
    motorcycle_info["Stroke (mm)"] = pd.to_numeric(
        motorcycle_info["Stroke (mm)"].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )

motorcycle_info = motorcycle_info.dropna()
motorcycle_info_num = motorcycle_info.select_dtypes(include="number")

print("Cleaned shape:", motorcycle_info.shape)
motorcycle_info_num.head()


In [ ]:
summary_stats = motorcycle_info_num.describe()
correlation_matrix = motorcycle_info_num.corr()

print("Numeric columns:", list(motorcycle_info_num.columns))
summary_stats


In [ ]:
correlation_matrix


### Exercise 8.1: Filter with Multiple Conditions

Student prompt: create one filtered subset using at least three conditions.

Suggested practice target: Yamaha sport motorcycles with displacement greater than 950 ccm.

Before coding, identify the brand, category, and displacement columns; decide whether text comparisons need capitalization handling; then combine conditions with `&`.


In [ ]:
filtered_motorcycles = motorcycle_info[
    (motorcycle_info["Brand"].str.lower() == "yamaha")
    & (motorcycle_info["Category"].str.contains("sport", case=False, na=False))
    & (motorcycle_info["Displacement (ccm)"] > 950)
]

# Backward-compatible alias for older TA notes.
yamaha_sport_large = filtered_motorcycles

filtered_motorcycles[["Brand", "Model", "Category", "Displacement (ccm)"]].head()


**TA checkpoint:** With the current cleaning step, this filter returns `7` rows. Accept equivalent filters that handle capitalization and include sport-touring categories if students explain their choice. If students get zero rows, have them inspect capitalization and category values separately before combining conditions.


### Exercise 8.2: Visualize Brand Counts

Student prompt: create a horizontal bar chart of motorcycle counts by brand.

Guiding questions:

1. Should you plot every brand or only the most common brands?
2. Which pandas method counts categories?
3. Which chart direction makes long brand names readable?
4. What title and axis labels will make the plot clear?


In [ ]:
brand_counts = motorcycle_info["Brand"].value_counts().head(15)

ax = brand_counts.sort_values().plot(kind="barh", figsize=(8, 6))
ax.set_title("Top 15 Motorcycle Brands in the Cleaned Dataset")
ax.set_xlabel("Number of motorcycles")
ax.set_ylabel("Brand")
plt.tight_layout()
plt.show()


## Section 9: K-means Preview

K-means is unsupervised learning: it groups rows without known labels. In Assignment 3, students apply it to numeric motorcycle features.

**TA note:** The student version asks them to choose columns and complete the K-means steps. The solution below uses four numeric columns as a reasonable starting point, but students should understand that feature choice, scaling, and `k` affect the clusters.


In [ ]:
cluster_columns = ["Displacement (ccm)", "Torque (Nm)", "Power (hp)", "Dry weight (kg)"]
cluster_data = motorcycle_info[cluster_columns].dropna()

kmeans = KMeans(n_clusters=3, random_state=1, n_init=10)
clusters = kmeans.fit_predict(cluster_data)

motorcycle_info_with_clusters = motorcycle_info.loc[cluster_data.index].copy()
motorcycle_info_with_clusters["cluster"] = clusters
motorcycle_info_with_clusters[["Brand", "Model", "Category", *cluster_columns, "cluster"]].head()


### Exercise 9.1: Interpret Clusters Carefully

Student prompt: after running K-means, use cluster counts and feature averages to describe one cluster cautiously.

**TA hint:** Ask students to support the interpretation with `groupby("cluster")[cluster_columns].mean()` rather than naming clusters from intuition alone.


In [ ]:
cluster_counts = motorcycle_info_with_clusters["cluster"].value_counts().sort_index()
cluster_summary = motorcycle_info_with_clusters.groupby("cluster")[cluster_columns].mean().round(1)

print("Cluster counts:")
print(cluster_counts)
cluster_summary


**Answer notes:** One reasonable interpretation is that cluster `0` contains larger/heavier motorcycles on average, cluster `1` contains smaller/lighter motorcycles, and cluster `2` sits between them with higher power than cluster `1`. Students should avoid treating cluster labels as true categories; K-means groups by the selected numeric features and can change if features, scaling, or `k` change.


## Assignment 3 Checklist

Before submitting Assignment 3, check that the notebook includes CSV loading, inspection, cleaning, numeric summaries, correlations, a filtered result, labeled plots, K-means clustering, and short interpretations.


## Reflection Answer Notes

1. Supervised tasks learn from known answers `y`; unsupervised tasks find structure without answer labels.
2. Test data stays separate so evaluation uses unseen examples.
3. Common error-prone steps: file paths, cleaning numeric-looking text columns, dropping too many rows, or interpreting clusters too strongly.
4. Avoid claiming one score or cluster plot proves a universal result.
